In [5]:
import random
import numpy as np
import pandas as pd
import h5py
from camera import Camera

In [19]:
def make_samples(cam, n_bin, drop_rate, add_rate, 
                 label, n_samples, seed, sigma, 
                 roll_rate, hr_to_idx):
    cam.id_point(label, 0)
    res = [360, 360]
    focal = 671
    max_radi = np.sqrt((res[0]**2) + (res[1]**2)) / 2 
    bins = np.linspace(0, max_radi, n_bin + 1)

    rng = np.random.default_rng(seed)

    for n in range(n_samples):
        cam.id_point(label, roll_rate * n)

        centroids = cam.create_centroids(focal, res, None, None) # centroids is pd df
        centroids = centroids[centroids["hr"].ne(label)]
    
        drop_n_stars = int(centroids["hr"].size * drop_rate)
        add_n_stars = int(centroids["hr"].size * add_rate)
        
        coords = np.array(centroids[["px", "py"]])
        coords += rng.normal(0, sigma, coords.shape)

        visible = ((coords[:, 0] >= 0) & (coords[:, 0] < res[0]) &
                   (coords[:, 1] >= 0) & (coords[:, 1] < res[1]))
        coords = coords[visible] 

        drop_count = min(len(coords), int(round(len(coords) * drop_rate)))
        if drop_count:
            keep = np.ones(len(coords), dtype=bool)
            keep[rng.choice(len(coords), size=drop_count, replace=False)] = False
            coords = coords[keep]

        add_count = int(round(len(coords) * add_rate))
        if add_count:
            false_coords = np.column_stack(
                (
                    rng.uniform(0.0, res[0], size=add_count),
                    rng.uniform(0.0, res[1], size=add_count),
                )
            )
            coords = np.vstack((coords, false_coords))

        distances = np.linalg.norm(coords - center, axis=1)
        histo = np.histogram(distances, bins)
        result = histo[0].astype(np.float32)
        result /= max(result.sum(), 1.0)
        
        with h5py.File("data.hdf5", "a") as f:
            mapped_label = hr_to_idx[label]
            if str(mapped_label) not in f:
                grp = f.create_group(str(mapped_label))
            else:
                grp = f[str(mapped_label)]
            
            grp.create_dataset(f"sample_{n}", data=result)

In [20]:
seed = 1
random.seed(seed)

data_path = "./hygdata_v42.csv"
data = pd.read_csv(data_path)
data.drop_duplicates(subset="hr", inplace=True)
mask = data["hr"].notna()
cam = Camera(data[mask])

In [21]:
#some hr ids are missing so indexes are not necessarily correct
unique_hr_ids = sorted(set(hr for hr in data["hr"]))
hr_to_idx = {hr_id: i for i, hr_id in enumerate(unique_hr_ids)}
idx_to_hr = {i: hr_id for hr_id, i in hr_to_idx.items()} 

In [ ]:
s = 0
for index, row in data[mask].iterrows():
    label = row["hr"]
    print(f"Making data for hr:{label} index: {hr_to_idx[label]}!")
    roll_rate = 1
    make_samples(cam, 25, 0.0, 0.0, label, 
                 360, seed, 0, roll_rate, hr_to_idx)    
    s += 1
print("Done!")
print(f"{s} Stars accounted for!")

Making data for hr:9077.0 index: 8995!
Making data for hr:9078.0 index: 8996!
Making data for hr:9079.0 index: 8997!
Making data for hr:9080.0 index: 8998!
Making data for hr:9081.0 index: 8999!
Making data for hr:9083.0 index: 9001!
Making data for hr:9082.0 index: 9000!
Making data for hr:9084.0 index: 9002!
Making data for hr:9085.0 index: 9003!
Making data for hr:9086.0 index: 9004!
Making data for hr:9087.0 index: 9005!
Making data for hr:9089.0 index: 9007!
Making data for hr:9088.0 index: 9006!
Making data for hr:9091.0 index: 9009!
Making data for hr:9092.0 index: 9010!
Making data for hr:9093.0 index: 9011!
Making data for hr:9094.0 index: 9012!
Making data for hr:9095.0 index: 9013!
Making data for hr:9096.0 index: 9014!
Making data for hr:9097.0 index: 9015!
Making data for hr:9098.0 index: 9016!
Making data for hr:9099.0 index: 9017!
Making data for hr:9100.0 index: 9018!
Making data for hr:9101.0 index: 9019!
Making data for hr:9102.0 index: 9020!
Making data for hr:9103.0